# Steps 7 & 8: Anomaly Segmentation Evaluation

This notebook evaluates both **Pixel-based** (ERFNet) and **Mask-based** (EoMT) models for anomaly segmentation across several benchmarks, including SMIYC and Fishyscapes.

## Objectives:
1. **Evaluate ERFNet (Pixel-based):** MSP, MaxLogit, and Max Entropy.
2. **Evaluate EoMT (Mask-based):** MSP, MaxLogit, Max Entropy, and RbA across 3 checkpoints.
3. **Temperature Scaling Search:** Optimize MSP scores using the cached "Smart Trick" logic.
4. **Generate Results Tables:** Produce the exact tables requested in the project guide.

In [ ]:
!pip install ood_metrics

In [ ]:
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null
!pip install gitignore_parser > /dev/null
!pip install lightning > /dev/null

In [ ]:
# @title Setup & Imports
import os
import sys
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/FundGitHubProject/'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

eomt_path = os.path.join(PROJECT_ROOT, 'eomt')
if eomt_path not in sys.path:
    sys.path.insert(0, eomt_path)

if not os.path.exists('/content/Fundamental_Project'):
    !ln -s {PROJECT_ROOT} /content/Fundamental_Project

%cd /content/Fundamental_Project

from posthoc_metrics import (
    get_pixel_msp, get_pixel_max_logit, get_pixel_entropy,
    get_mask_msp, get_mask_max_logit, get_mask_entropy, get_mask_rba,
    compute_metrics, cache_model_outputs, fast_temperature_search
)
from eval.Validation_Dataset import anomaly_datasets
from eomt.checkpoint_utils import get_finetuned_model
from eval.erfnet import ERFNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/FundGitHubProject
Using device: cuda


In [ ]:
# @title Dataset Setup
datasets_config = {
    'SMIYC RA-21': 'RoadAnomaly21',
    'SMIYC RO-21': 'RoadObstacle21',
    'FS L&F': 'FS_LostFound',
    'FS Static': 'FS_Static',
    'Road Anomaly': 'RoadAnomaly'
}

dataloaders = {}
for display_name, internal_name in datasets_config.items():
    dm = anomaly_datasets.AnomalyDataModule(dataset_name=internal_name, img_size=(640, 640))
    dm.setup()
    dataloaders[display_name] = dm.val_dataloader()

## 🛠️ Step 7: Pixel-based Baselines (ERFNet)
Evaluating ERFNet with pixel-wise scoring methods.

In [ ]:
def evaluate_pixel_model(model, dataloader, method_name):
    model.eval()
    all_scores, all_gts = [], []

    scoring_fns = {
        'MSP': get_pixel_msp,
        'MaxLogit': lambda x: get_pixel_max_logit(x),
        'Max Entropy': get_pixel_entropy
    }
    fn = scoring_fns[method_name]

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"ERFNet {method_name}"):
            img, gt = batch['image'].to(device), batch['label']
            logits = model(img)
            score_map = fn(logits)

            # Resize score map to match GT resolution if needed
            if score_map.shape[-2:] != gt.shape[-2:]:
                score_map = torch.nn.functional.interpolate(score_map.unsqueeze(1), size=gt.shape[-2:], mode='bilinear').squeeze(1)

            all_scores.append(score_map.cpu().numpy().flatten())
            all_gts.append(gt.numpy().flatten())

    return compute_metrics(np.concatenate(all_scores), np.concatenate(all_gts))

# 1. Load ERFNet
#
erfnet = ERFNet(num_classes=20).to(device)
# slicing
weights_path = 'trained_models/erfnet_pretrained.pth' # Update if needed
if os.path.exists(weights_path):
    state_dict = torch.load(weights_path, map_location=device)
    erfnet.load_state_dict(state_dict.get('state_dict', state_dict), strict=False)

# 2. Run Evaluation
pixel_results = []
for method in ['MSP', 'MaxLogit', 'Max Entropy']:
    row = {'Model': 'ERFNet', 'Method': method}
    for ds_name, loader in dataloaders.items():
        metrics = evaluate_pixel_model(erfnet, loader, method)
        row[f"{ds_name} AuPRC"] = metrics['auprc']
        row[f"{ds_name} FPR95"] = metrics['fpr95']
    pixel_results.append(row)

df_pixel = pd.DataFrame(pixel_results)
df_pixel

ERFNet Max Entropy: 100%|██████████| 60/60 [00:02<00:00, 27.37it/s]


,Model,Method,SMIYC RA-21 AuPRC,SMIYC RA-21 FPR95,SMIYC RO-21 AuPRC,SMIYC RO-21 FPR95,FS L&F AuPRC,FS L&F FPR95,FS Static AuPRC,FS Static FPR95,Road Anomaly AuPRC,Road Anomaly FPR95
0,ERFNet,MSP,15.197344,93.586960,0.728872,93.944031,0.276551,95.346854,1.407932,94.183727,10.020039,94.192503
1,ERFNet,MaxLogit,15.103038,93.936857,0.742766,94.725714,0.276619,95.388963,1.420474,94.344728,9.984591,94.276491
2,ERFNet,Max Entropy,14.669373,95.007448,0.694149,95.105971,0.280250,95.132765,1.366936,94.820535,9.773233,94.987372


## 🛠️ Step 8: Mask-based Baselines (EoMT)
Evaluating EoMT across three checkpoints with query-based and RbA scoring.

In [ ]:
def evaluate_mask_model(model, dataloader, method_name):
    model.eval()
    all_scores, all_gts = [], []

    scoring_fns = {
        'MSP': get_mask_msp,
        'MaxLogit': lambda m_cls, m_pred: get_mask_max_logit(m_cls, m_pred),
        'Max Entropy': get_mask_entropy,
        'RbA': get_mask_rba
    }
    fn = scoring_fns[method_name]

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"EoMT {method_name}"):
            img, gt = batch['image'].to(device), batch['label']
            m_pred, m_cls = model(img) # EoMT returns (mask_pred_list, class_pred_list)

            # Use last layer outputs
            score_map = fn(m_cls[-1], m_pred[-1])

            if score_map.shape[-2:] != gt.shape[-2:]:
                score_map = torch.nn.functional.interpolate(score_map.unsqueeze(1), size=gt.shape[-2:], mode='bilinear').squeeze(1)

            all_scores.append(score_map.cpu().numpy().flatten())
            all_gts.append(gt.numpy().flatten())

    return compute_metrics(np.concatenate(all_scores), np.concatenate(all_gts))

checkpoints = {
    'COCO': 'checkpoints/eomt_coco.ckpt',
    'Cityscapes': 'checkpoints/eomt_cityscapes.ckpt',
    'Fine-tuned': 'checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt'
}

mask_results = []
for ckpt_name, ckpt_path in checkpoints.items():
    if not os.path.exists(ckpt_path): continue
    model = get_finetuned_model(ckpt_path).to(device)

    for method in ['MSP', 'MaxLogit', 'Max Entropy', 'RbA']:
        row = {'Model': f'EoMT ({ckpt_name})', 'Method': method}
        for ds_name, loader in dataloaders.items():
            metrics = evaluate_mask_model(model, loader, method)
            row[f"{ds_name} AuPRC"] = metrics['auprc']
            row[f"{ds_name} FPR95"] = metrics['fpr95']
        mask_results.append(row)

    del model; torch.cuda.empty_cache()

df_mask = pd.DataFrame(mask_results)
df_mask

--- Initializing Enhanced Architecture ---
Blocks: 3 | LoRA R: 8 | Backbone: vit_base_patch14_reg4_dinov2
Loading weights from: checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt
NOTE: Unexpected keys: 1
✅ Model ready for inference.


EoMT RbA: 100%|██████████| 60/60 [00:15<00:00,  3.78it/s]


,Model,Method,SMIYC RA-21 AuPRC,SMIYC RA-21 FPR95,SMIYC RO-21 AuPRC,SMIYC RO-21 FPR95,FS L&F AuPRC,FS L&F FPR95,FS Static AuPRC,FS Static FPR95,Road Anomaly AuPRC,Road Anomaly FPR95
0,EoMT (Fine-tuned),MSP,72.997016,20.058957,74.068477,0.729912,51.437018,16.958431,78.799100,6.275098,71.649358,29.614434
1,EoMT (Fine-tuned),MaxLogit,63.476194,25.784082,82.655580,0.314602,7.039712,20.229243,49.026430,6.262528,63.336373,63.000124
2,EoMT (Fine-tuned),Max Entropy,74.397010,38.798678,3.222360,81.733453,0.547357,52.633479,2.664323,77.597731,13.776233,63.073550
3,EoMT (Fine-tuned),RbA,41.322537,96.872729,70.587805,1.060411,47.827526,61.347983,82.514676,2.450075,65.713328,78.206962


## 🌡️ Temperature Scaling (Smart Trick)
Optimizing MSP calibration using cached logits.

In [ ]:
import importlib
import sys

# Fix the legacy import directly in the source file
file_path = '/content/drive/MyDrive/FundGitHubProject/posthoc_metrics/fast_eval_utils.py'
with open(file_path, 'r') as f:
    code = f.read()
code = code.replace('get_msp_anomaly_map', 'get_mask_msp')
with open(file_path, 'w') as f:
    f.write(code)

# Reload the modules to pick up the change
if 'posthoc_metrics.fast_eval_utils' in sys.modules:
    importlib.reload(sys.modules['posthoc_metrics.fast_eval_utils'])
if 'posthoc_metrics' in sys.modules:
    importlib.reload(sys.modules['posthoc_metrics'])
from posthoc_metrics import fast_temperature_search

# Load best model for calibration (e.g., Fine-tuned EoMT on Road Anomaly)
model = get_finetuned_model(checkpoints['Fine-tuned']).to(device)
calib_loader = dataloaders['Road Anomaly']

# Wrapper to convert dict batches to (image, target) tuples for the caching function
class TupleWrapperLoader:
    def __init__(self, loader):
        self.loader = loader
    def __iter__(self):
        for batch in self.loader:
            yield batch['image'], batch['label']
    def __len__(self):
        return len(self.loader)

CACHE_DIR = 'temp_scaling_cache'
cache_model_outputs(model, TupleWrapperLoader(calib_loader), CACHE_DIR, device=device)

temps = [0.5, 0.75, 1.0, 1.1]
search_results = fast_temperature_search(CACHE_DIR, scoring_fn=get_mask_msp, temperatures=temps)

# Find best t
best_t = max(search_results, key=lambda t: search_results[t]['auprc'])
best_metrics = search_results[best_t]

# Construct Table
temp_table = []
for t in temps:
    temp_table.append({'Method': f'MSP (t={t})', 'AuPRC': search_results[t]['auprc'], 'FPR95': search_results[t]['fpr95']})
temp_table.append({'Method': f'MSP (best t={best_t})', 'AuPRC': best_metrics['auprc'], 'FPR95': best_metrics['fpr95']})

df_temp = pd.DataFrame(temp_table)
df_temp

--- Initializing Enhanced Architecture ---
Blocks: 3 | LoRA R: 8 | Backbone: vit_base_patch14_reg4_dinov2
Loading weights from: checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt
NOTE: Unexpected keys: 1
✅ Model ready for inference.
--- Caching outputs to temp_scaling_cache ---


100%|██████████| 60/60 [00:27<00:00,  2.21it/s]


Testing Temperature T=0.5...


IndexError: boolean index did not match indexed array along dimension 0; dimension is 1536000 but corresponding boolean dimension is 24576000